In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import xarray as xr

from fast_o3_iv import utils, metrics
import fast_o3_iv.bias_correction as bc
from fast_o3_iv.config import INTERIM_DATA_DIR
from fast_o3_iv.modeling.train import DatasetConfig


In [ ]:
# load observation osdma8
ramp_osdma8 = xr.open_dataset((
        INTERIM_DATA_DIR
        / "RAMP"
        / "Jacob_wRAMP_OSDMA8_1990_2017.nc"))


In [ ]:
# load modeled monthly mda8o3
config_hist = DatasetConfig(
            source_id = 'CESM2-WACCM6',
            cmip_experiment_id = 'historical',
            variable_setname = 'setA',
            year_range = range(1975,2015)
        )
wa6_hist = utils.load_or_create_results_ds(
    "testing",
    "GPR-local",
    config_hist,
)

config_ssp245 = DatasetConfig(
            source_id = 'CESM2-WACCM6',
            cmip_experiment_id = 'ssp245',
            variable_setname = 'setA',
            year_range = range(2015,2055)
        )
wa6_ssp245 = utils.load_or_create_results_ds(
    "testing",
    "GPR-local",
    config_ssp245,
)

Loading existing dataset from /home/eleroy/proj-dirs/fast-o3-iv/data/processed/model_outputs/results/testing/NCAR/CESM2-WACCM6/historical/all/setA/GPR-local/NCAR.CESM2-WACCM6.historical.all.setA.testing.GPR-local.nc
Loading existing dataset from /home/eleroy/proj-dirs/fast-o3-iv/data/processed/model_outputs/results/testing/NCAR/CESM2-WACCM6/ssp245/all/setA/GPR-local/NCAR.CESM2-WACCM6.ssp245.all.setA.testing.GPR-local.nc


In [ ]:
# convert monthly mda8o3 to osdma8 

# add climatology (only required if target == mda8o3_anm (setC))
#wa6_hist = utils.add_mda8o3_climatology(wa6_hist, config) 
#wa6_ssp245 = utils.add_mda8o3_climatology(wa6_ssp245, config) 

wa6_hist_osdma8 = metrics.calc_osdma8(wa6_hist)
wa6_ssp245_osdma8 = metrics.calc_osdma8(wa6_ssp245)

In [ ]:
# calculate scaling factor based on historical mod vs obs osdma8
scaling_factor, obs_base, mod_base = bc.get_bc_scaling_factor(wa6_hist_osdma8, ramp_osdma8)

In [ ]:
# apply bias correction
wa6_hist_osdma8_bc = bc.bias_correct(wa6_hist_osdma8, scaling_factor)
wa6_ssp245_osdma8_bc = bc.bias_correct(wa6_ssp245_osdma8, scaling_factor)

In [ ]:
# save to netcdf
file_path = bc.create_osdma8_bc_save_path(config_hist, "GPR-local", "testing")
wa6_hist_osdma8_bc.to_netcdf(file_path)

file_path = bc.create_osdma8_bc_save_path(config_ssp245, "GPR-local", "testing")
wa6_ssp245_osdma8_bc.to_netcdf(file_path)